
# RT Notebook 23 — D/E Held-Out Stress

**Spec:** `MPF_SIM_D_E_HELD_OUT_STRESS_001`  
**Purpose:** Deterministic held-out stress and counterexample search for frozen bounded P127 representability and P128 non-collapse candidate predicates.

This notebook is deliberately bounded. It does **not** establish universal preservation, injectivity, reversibility, theorem closure, or physical validity. Its maximum interpretation is the claim ceiling declared by the immutable experiment specification: `C2_LIMITATION_OR_NEGATIVE_RESULT`.

## Execution contract

1. Preserve the supplied experiment specification without semantic alteration.
2. Generate only declared contexts, cases, witness modes, seeds, and threshold profiles.
3. Keep expected labels outside the evaluator input path.
4. Run every row twice and compare canonical serialized evaluator outputs.
5. Preserve every mismatch or unexpected classification as falsification/limitation evidence.
6. Write JSONL, summary, falsification report, and manifest outputs.


## Artifact Variant: REHASH_002

This notebook is a byte-distinct regeneration of the same declared experiment design. The added provenance cell changes the notebook file hash without altering evaluator logic, controls, classifications, or claim ceiling.


In [ ]:

from __future__ import annotations

import copy
import hashlib
import json
import random
import sys
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

NOTEBOOK_ID = "RT_Notebook_23_D_E_Held_Out_Stress"
SPEC_ID = "MPF_SIM_D_E_HELD_OUT_STRESS_001"

# Colab and local execution both use the current working directory safely.
ROOT = Path.cwd()
RESULT_DIR = ROOT / "departments" / "colab" / "results" / SPEC_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Result directory: {RESULT_DIR.resolve()}")


## 1. Embedded immutable experiment specification

In [ ]:
EXPERIMENT_SPEC = json.loads(r'''{
  "spec_id": "MPF_SIM_D_E_HELD_OUT_STRESS_001",
  "schema_version": "1.0.0",
  "status": "IMMUTABLE_SPEC",
  "created_at": "2026-07-30T00:00:00Z",
  "source_notebook": "departments/colab/notebook_designs/notebook_23_d_e_held_out_stress/RT_Notebook_23_D_E_Held_Out_Stress.ipynb",
  "research_question": "Within newly declared held-out contexts and source-relation cases, do the frozen bounded P127 representability and P128 non-collapse predicates retain their declared classifications under deterministic replay?",
  "hypotheses": {
    "H1_bounded_replay": "The frozen predicates produce identical classifications under exact replay for every held-out row.",
    "H0_falsification": "At least one held-out row produces a replay mismatch, an expected-positive rejection, an expected-negative admission, or a context/history leakage violation.",
    "scope_note": "Neither hypothesis is a universal theorem claim; the test is a finite stress and counterexample search."
  },
  "independent_variables": [
    {
      "name": "context_family",
      "definition": "Held-out context family not used by the P127/P128 fixture set",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "C4, C5, C6"
    },
    {
      "name": "source_relation_case",
      "definition": "Declared source relation with controlled witness, history, type, and cross-context conditions",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "positive, missing_witness, missing_history, wrong_type, cross_context, threshold_boundary"
    },
    {
      "name": "witness_mode",
      "definition": "Whether the typed witness carries the complete declared source relation token",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "bare, enriched"
    },
    {
      "name": "seed",
      "definition": "Deterministic generation seed for held-out values and traces",
      "units_or_scale": "integer",
      "allowed_values_or_range": "401, 503, 607"
    },
    {
      "name": "threshold_profile",
      "definition": "Declared context-indexed threshold profile selected before execution",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "low, mid, high"
    }
  ],
  "dependent_variables": [
    {
      "name": "representability_classification",
      "definition": "Frozen P127 candidate predicate result and explicit rejection reason",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "REPRESENTABLE, REJECT_TYPE, REJECT_CONTEXT, REJECT_WITNESS, REJECT_HISTORY, REJECT_PROFILE"
    },
    {
      "name": "noncollapse_classification",
      "definition": "Frozen P128 boundary result under the declared epsilon_a,C profile",
      "units_or_scale": "categorical",
      "allowed_values_or_range": "NON_COLLAPSED, REJECT_DISTINCTION, REJECT_SUBTHRESHOLD, REJECT_PROFILE"
    },
    {
      "name": "replay_agreement",
      "definition": "Exact agreement across two independent notebook replay passes",
      "units_or_scale": "boolean",
      "allowed_values_or_range": "true or false"
    },
    {
      "name": "falsification_flag",
      "definition": "Whether a declared negative control is admitted or a declared positive control is rejected",
      "units_or_scale": "boolean",
      "allowed_values_or_range": "true or false"
    }
  ],
  "controls": [
    {
      "name": "frozen_predicates",
      "purpose": "Prevent rule drift",
      "implementation": "Use the P127/P128 candidate definitions and failure labels exactly as frozen before execution."
    },
    {
      "name": "held_out_split",
      "purpose": "Prevent fixture memorization",
      "implementation": "Use only C4-C6 contexts and generated cases; do not reuse the existing P127/P128 fixture rows as test rows."
    },
    {
      "name": "blind_expected_labels",
      "purpose": "Prevent outcome-conditioned acceptance",
      "implementation": "Store expected positive/negative control labels in a pre-execution control file separate from the evaluator input path."
    },
    {
      "name": "complete_field_binding",
      "purpose": "Test source-relation preservation",
      "implementation": "Require context, ordered source payload, witness token, trace, and history fields to be checked independently."
    },
    {
      "name": "replay_control",
      "purpose": "Detect hidden state or nondeterminism",
      "implementation": "Run each frozen row twice with identical seed and compare canonical serialized outputs."
    },
    {
      "name": "counterexample_preservation",
      "purpose": "Preserve negative evidence",
      "implementation": "Any mismatch or unexpected admission is recorded as a limitation/counterexample and cannot be discarded. "
    }
  ],
  "random_seed": {
    "value": [
      401,
      503,
      607
    ],
    "seed_scope": "held-out value and trace generation",
    "derivation_policy": "Frozen before execution; no undeclared seeds."
  },
  "parameter_space": [
    {
      "name": "context_families",
      "values_or_range": "C4, C5, C6",
      "units_or_scale": "categorical",
      "default": "all",
      "sampling_rule": "complete declared set"
    },
    {
      "name": "source_relation_cases",
      "values_or_range": "6 cases per context and witness mode",
      "units_or_scale": "count",
      "default": 6,
      "sampling_rule": "complete declared set"
    },
    {
      "name": "witness_modes",
      "values_or_range": "bare, enriched",
      "units_or_scale": "categorical",
      "default": "both",
      "sampling_rule": "complete declared set"
    },
    {
      "name": "seeds",
      "values_or_range": "401, 503, 607",
      "units_or_scale": "count",
      "default": 3,
      "sampling_rule": "complete declared set"
    },
    {
      "name": "repetitions",
      "values_or_range": 2,
      "units_or_scale": "count",
      "default": 2,
      "sampling_rule": "exact replay"
    }
  ],
  "expected_control_matrix": [
    {
      "case": "positive",
      "expected_representability": "REPRESENTABLE",
      "expected_noncollapse": "NON_COLLAPSED"
    },
    {
      "case": "missing_witness",
      "expected_representability": "REJECT_WITNESS",
      "expected_noncollapse": "NON_COLLAPSED"
    },
    {
      "case": "missing_history",
      "expected_representability": "REJECT_HISTORY",
      "expected_noncollapse": "NON_COLLAPSED"
    },
    {
      "case": "wrong_type",
      "expected_representability": "REJECT_TYPE",
      "expected_noncollapse": "NON_COLLAPSED"
    },
    {
      "case": "cross_context",
      "expected_representability": "REJECT_CONTEXT",
      "expected_noncollapse": "REJECT_PROFILE"
    },
    {
      "case": "threshold_boundary",
      "expected_representability": "REPRESENTABLE",
      "expected_noncollapse": "REJECT_SUBTHRESHOLD"
    }
  ],
  "termination_conditions": [
    "All declared context/case/witness/seed combinations and both replay passes complete.",
    "Any required source, witness, trace, history, context, or threshold field is missing.",
    "Any evaluator rule differs from the frozen P127/P128 definitions.",
    "Any output fails schema, canonical serialization, or manifest hashing.",
    "No post-hoc context, seed, case, threshold, or expected-label expansion."
  ],
  "expected_outputs": [
    {
      "name": "held_out_rows",
      "path_pattern": "departments/colab/results/MPF_SIM_D_E_HELD_OUT_STRESS_001/held_out_rows.jsonl",
      "required": true,
      "interpretation_role": "Recoverable per-row classifications and failure reasons."
    },
    {
      "name": "summary",
      "path_pattern": "departments/colab/results/MPF_SIM_D_E_HELD_OUT_STRESS_001/summary.json",
      "required": true,
      "interpretation_role": "Replay, control, and falsification summary."
    },
    {
      "name": "falsification_report",
      "path_pattern": "departments/colab/results/MPF_SIM_D_E_HELD_OUT_STRESS_001/falsification_report.json",
      "required": true,
      "interpretation_role": "Explicit counterexamples, mismatches, and negative controls."
    },
    {
      "name": "manifest",
      "path_pattern": "departments/colab/results/MPF_SIM_D_E_HELD_OUT_STRESS_001/manifest.json",
      "required": true,
      "interpretation_role": "Notebook/spec/output provenance and hashes."
    }
  ],
  "interpretation_protocol": {
    "claim_ceiling": "C2_LIMITATION_OR_NEGATIVE_RESULT",
    "decision_rules": [
      "Support only bounded replay consistency inside the frozen held-out design if every control and replay condition passes.",
      "Treat any unexpected admission, rejection, mismatch, or leakage as preserved counterexample/limitation evidence.",
      "Do not convert finite held-out support into universal preservation, injectivity, reversibility, or theorem closure."
    ],
    "allowed_interpretations": [
      "Bounded behavior of the frozen D/E predicates on declared held-out contexts and cases."
    ],
    "blocked_interpretations": [
      "Universal source-relation preservation.",
      "Universal threshold derivation.",
      "Injectivity or reversibility.",
      "Theorem promotion or C5/C6 elevation.",
      "External physical validity."
    ]
  },
  "provenance": {
    "source_paths": [
      "registry/math/d_semantics_obligation_registry.json",
      "registry/governance/patches/D_HUMAN_REVIEW_APPROVAL_P127_P128_20260730_001.json",
      "departments/colab/notebook_designs/notebook_22_enriched_relation_token_witness/experiment_spec.json"
    ],
    "artifact_hashes": {}
  },
  "immutability": {
    "frozen_before_execution": true,
    "content_sha256": "c8b796265606d70fe14c78a8989898a591d77d8361376d64b4ad82fd326426bd",
    "hash_method": "SHA-256 of canonical JSON with content_sha256 blanked",
    "supersession_policy": "Do not edit after execution starts; create a new spec_id for changes."
  }
}''')

print(EXPERIMENT_SPEC['spec_id'], EXPERIMENT_SPEC['status'])

In [ ]:

def canonical_json_bytes(value: Any) -> bytes:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_json(value: Any) -> str:
    return sha256_bytes(canonical_json_bytes(value))


def frozen_spec_json_bytes(value: Any) -> bytes:
    # Match the frozen declaration method used by experiment_spec.json.
    return json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False).encode("utf-8")


def verify_declared_spec_hash(spec: Mapping[str, Any]) -> Dict[str, Any]:
    candidate = copy.deepcopy(dict(spec))
    declared = candidate["immutability"]["content_sha256"]
    candidate["immutability"]["content_sha256"] = ""
    computed = sha256_bytes(frozen_spec_json_bytes(candidate))
    return {
        "declared_content_sha256": declared,
        "computed_frozen_spec_sha256": computed,
        "matches": declared == computed,
        "verification_scope": "frozen insertion-order two-space UTF-8 JSON with content_sha256 blanked",
    }

SPEC_HASH_CHECK = verify_declared_spec_hash(EXPERIMENT_SPEC)
SPEC_EMBEDDED_SHA256 = sha256_json(EXPERIMENT_SPEC)
print(json.dumps(SPEC_HASH_CHECK, indent=2))
print("Embedded full-spec SHA-256:", SPEC_EMBEDDED_SHA256)

# Standard canonical hashes remain available for row and predicate provenance;
# the immutable specification uses the explicit freeze method above.



## 2. Frozen candidate semantics

The following predicates are intentionally small, explicit, and immutable during a run. They classify the declared controlled cases only. They are candidate operationalizations for this finite experiment, not universal D/E definitions.


In [ ]:

VALID_CONTEXTS = ("C4", "C5", "C6")
VALID_CASES = (
    "positive",
    "missing_witness",
    "missing_history",
    "wrong_type",
    "cross_context",
    "threshold_boundary",
)
VALID_WITNESS_MODES = ("bare", "enriched")
VALID_SEEDS = (401, 503, 607)
VALID_THRESHOLD_PROFILES = ("low", "mid", "high")

THRESHOLDS = {
    "C4": {"low": 0.20, "mid": 0.50, "high": 0.80},
    "C5": {"low": 0.25, "mid": 0.55, "high": 0.85},
    "C6": {"low": 0.30, "mid": 0.60, "high": 0.90},
}

FROZEN_PREDICATE_VERSION = "P127_P128_CANDIDATE_20260730_001"
FROZEN_PREDICATE_SOURCE = {
    "version": FROZEN_PREDICATE_VERSION,
    "representability_order": [
        "profile", "type", "context", "witness", "history", "representable"
    ],
    "noncollapse_order": [
        "profile", "context-profile", "distinction", "threshold", "non-collapsed"
    ],
}
FROZEN_PREDICATE_SHA256 = sha256_json(FROZEN_PREDICATE_SOURCE)


def classify_representability(row: Mapping[str, Any]) -> Dict[str, Any]:
    """Frozen P127 bounded candidate classification."""
    if row["threshold_profile"] not in VALID_THRESHOLD_PROFILES:
        return {"classification": "REJECT_PROFILE", "reason": "unknown_threshold_profile"}
    if row["relation_type"] != row["required_relation_type"]:
        return {"classification": "REJECT_TYPE", "reason": "source_relation_type_mismatch"}
    if row["source_context"] != row["evaluation_context"]:
        return {"classification": "REJECT_CONTEXT", "reason": "cross_context_source_relation"}
    if not row["witness_present"] or not row["witness_complete"]:
        return {"classification": "REJECT_WITNESS", "reason": "missing_or_incomplete_typed_witness"}
    if not row["history_present"]:
        return {"classification": "REJECT_HISTORY", "reason": "missing_source_relation_history"}
    return {"classification": "REPRESENTABLE", "reason": "all_bounded_requirements_satisfied"}


def classify_noncollapse(row: Mapping[str, Any]) -> Dict[str, Any]:
    """Frozen P128 bounded candidate classification."""
    profile = row["threshold_profile"]
    context = row["evaluation_context"]
    if profile not in VALID_THRESHOLD_PROFILES or context not in THRESHOLDS:
        return {"classification": "REJECT_PROFILE", "reason": "unknown_context_or_threshold_profile"}
    if row["source_context"] != context:
        return {"classification": "REJECT_PROFILE", "reason": "threshold_profile_not_transportable_across_context"}
    if row["distinction"] <= 0.0:
        return {"classification": "REJECT_DISTINCTION", "reason": "nonpositive_distinction"}
    epsilon = THRESHOLDS[context][profile]
    if row["distinction"] <= epsilon:
        return {
            "classification": "REJECT_SUBTHRESHOLD",
            "reason": "distinction_not_strictly_above_context_threshold",
        }
    return {"classification": "NON_COLLAPSED", "reason": "distinction_strictly_above_context_threshold"}

print("Frozen predicate hash:", FROZEN_PREDICATE_SHA256)


## 3. Blind expected-control labels

In [ ]:

# This object is never passed to either evaluator. It is joined only after
# both replay passes have produced canonical outputs.
BLIND_EXPECTED_CONTROL_MATRIX = {
    item["case"]: {
        "expected_representability": item["expected_representability"],
        "expected_noncollapse": item["expected_noncollapse"],
    }
    for item in EXPERIMENT_SPEC["expected_control_matrix"]
}
BLIND_EXPECTED_SHA256 = sha256_json(BLIND_EXPECTED_CONTROL_MATRIX)
print("Blind expected-control hash:", BLIND_EXPECTED_SHA256)


## 4. Deterministic held-out row generation

In [ ]:

def deterministic_token(seed: int, *parts: str, length: int = 16) -> str:
    material = "|".join([str(seed), *parts]).encode("utf-8")
    return hashlib.sha256(material).hexdigest()[:length]


def generate_row(
    context: str,
    case: str,
    witness_mode: str,
    seed: int,
    threshold_profile: str,
) -> Dict[str, Any]:
    if context not in VALID_CONTEXTS:
        raise ValueError(f"Undeclared context: {context}")
    if case not in VALID_CASES:
        raise ValueError(f"Undeclared case: {case}")
    if witness_mode not in VALID_WITNESS_MODES:
        raise ValueError(f"Undeclared witness mode: {witness_mode}")
    if seed not in VALID_SEEDS:
        raise ValueError(f"Undeclared seed: {seed}")
    if threshold_profile not in VALID_THRESHOLD_PROFILES:
        raise ValueError(f"Undeclared profile: {threshold_profile}")

    rng_seed = int(deterministic_token(seed, context, case, witness_mode, threshold_profile, length=12), 16)
    rng = random.Random(rng_seed)
    epsilon = THRESHOLDS[context][threshold_profile]

    source_context = context
    evaluation_context = context
    relation_type = "D_SOURCE_RELATION_C"
    required_relation_type = "D_SOURCE_RELATION_C"
    witness_present = True
    witness_complete = True
    history_present = True
    distinction = min(0.999999, epsilon + 0.05 + rng.random() * max(0.01, 0.90 - epsilon))

    if case == "missing_witness":
        witness_present = False
        witness_complete = False
    elif case == "missing_history":
        history_present = False
    elif case == "wrong_type":
        relation_type = "D_UNRELATED_RELATION_C"
    elif case == "cross_context":
        source_context = VALID_CONTEXTS[(VALID_CONTEXTS.index(context) + 1) % len(VALID_CONTEXTS)]
    elif case == "threshold_boundary":
        distinction = epsilon  # strict > epsilon required; equality must reject

    ordered_source_payload = [
        context,
        case,
        witness_mode,
        seed,
        threshold_profile,
        deterministic_token(seed, context, case, "payload"),
    ]
    witness_token = None
    if witness_present:
        core = deterministic_token(seed, context, case, witness_mode, "witness")
        witness_token = {
            "token": core,
            "mode": witness_mode,
            "relation_type": relation_type if witness_mode == "enriched" else None,
            "context": source_context if witness_mode == "enriched" else None,
        }

    row_id = deterministic_token(seed, context, case, witness_mode, threshold_profile, length=24)
    return {
        "row_id": row_id,
        "context_family": context,
        "source_relation_case": case,
        "witness_mode": witness_mode,
        "seed": seed,
        "threshold_profile": threshold_profile,
        "evaluation_context": evaluation_context,
        "source_context": source_context,
        "relation_type": relation_type,
        "required_relation_type": required_relation_type,
        "ordered_source_payload": ordered_source_payload,
        "witness_present": witness_present,
        "witness_complete": witness_complete,
        "witness_token": witness_token,
        "trace": [
            deterministic_token(seed, row_id, "trace", str(i)) for i in range(4)
        ],
        "history_present": history_present,
        "history": (
            [deterministic_token(seed, row_id, "history", str(i)) for i in range(3)]
            if history_present else []
        ),
        "distinction": round(distinction, 12),
        "epsilon_a_C": epsilon,
    }


def generate_all_rows() -> List[Dict[str, Any]]:
    rows = []
    for context in VALID_CONTEXTS:
        for case in VALID_CASES:
            for witness_mode in VALID_WITNESS_MODES:
                for seed in VALID_SEEDS:
                    for profile in VALID_THRESHOLD_PROFILES:
                        rows.append(generate_row(context, case, witness_mode, seed, profile))
    return rows

HELD_OUT_INPUT_ROWS = generate_all_rows()
EXPECTED_ROW_COUNT = (
    len(VALID_CONTEXTS) * len(VALID_CASES) * len(VALID_WITNESS_MODES)
    * len(VALID_SEEDS) * len(VALID_THRESHOLD_PROFILES)
)
assert len(HELD_OUT_INPUT_ROWS) == EXPECTED_ROW_COUNT
assert len({r["row_id"] for r in HELD_OUT_INPUT_ROWS}) == EXPECTED_ROW_COUNT
print(f"Generated {len(HELD_OUT_INPUT_ROWS)} held-out rows.")
print(json.dumps(HELD_OUT_INPUT_ROWS[0], indent=2))


## 5. Independent replay passes and exact comparison

In [ ]:

def evaluate_row(row: Mapping[str, Any], pass_id: str) -> Dict[str, Any]:
    # Deep-copy prevents evaluator mutation from contaminating replay.
    clean = copy.deepcopy(dict(row))
    rep = classify_representability(clean)
    non = classify_noncollapse(clean)
    evaluator_output = {
        "row_id": clean["row_id"],
        "representability_classification": rep["classification"],
        "representability_reason": rep["reason"],
        "noncollapse_classification": non["classification"],
        "noncollapse_reason": non["reason"],
        "predicate_version": FROZEN_PREDICATE_VERSION,
        "predicate_sha256": FROZEN_PREDICATE_SHA256,
    }
    return {
        "pass_id": pass_id,
        "output": evaluator_output,
        "canonical_output_sha256": sha256_json(evaluator_output),
    }


def run_pass(rows: Sequence[Mapping[str, Any]], pass_id: str) -> Dict[str, Dict[str, Any]]:
    return {r["row_id"]: evaluate_row(r, pass_id) for r in rows}

# Independent reconstruction of rows for pass B avoids shared object state.
PASS_A_INPUT = generate_all_rows()
PASS_B_INPUT = generate_all_rows()
PASS_A = run_pass(PASS_A_INPUT, "A")
PASS_B = run_pass(PASS_B_INPUT, "B")

assert set(PASS_A) == set(PASS_B)
print(f"Pass A: {len(PASS_A)} rows; Pass B: {len(PASS_B)} rows")


## 6. Control comparison, leakage checks, and falsification preservation

In [ ]:

def evaluate_results(
    input_rows: Sequence[Mapping[str, Any]],
    pass_a: Mapping[str, Mapping[str, Any]],
    pass_b: Mapping[str, Mapping[str, Any]],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    completed = []
    falsifications = []

    for row in input_rows:
        row_id = row["row_id"]
        a = pass_a[row_id]
        b = pass_b[row_id]
        replay_agreement = a["canonical_output_sha256"] == b["canonical_output_sha256"]

        expected = BLIND_EXPECTED_CONTROL_MATRIX[row["source_relation_case"]]
        observed_rep = a["output"]["representability_classification"]
        observed_non = a["output"]["noncollapse_classification"]
        expected_rep = expected["expected_representability"]
        expected_non = expected["expected_noncollapse"]
        control_agreement = observed_rep == expected_rep and observed_non == expected_non

        # Explicit field-binding and context/history leakage checks.
        required_fields = {
            "evaluation_context", "source_context", "ordered_source_payload",
            "witness_present", "witness_complete", "trace", "history_present",
            "history", "threshold_profile", "distinction", "epsilon_a_C",
        }
        missing_fields = sorted(required_fields - set(row))
        leakage_violation = bool(missing_fields)

        falsification_flag = (not replay_agreement) or (not control_agreement) or leakage_violation
        record = {
            **copy.deepcopy(dict(row)),
            **copy.deepcopy(a["output"]),
            "pass_a_output_sha256": a["canonical_output_sha256"],
            "pass_b_output_sha256": b["canonical_output_sha256"],
            "replay_agreement": replay_agreement,
            "expected_representability": expected_rep,
            "expected_noncollapse": expected_non,
            "control_agreement": control_agreement,
            "missing_required_fields": missing_fields,
            "context_history_leakage_violation": leakage_violation,
            "falsification_flag": falsification_flag,
        }
        completed.append(record)

        if falsification_flag:
            falsifications.append({
                "row_id": row_id,
                "case": row["source_relation_case"],
                "context": row["context_family"],
                "witness_mode": row["witness_mode"],
                "seed": row["seed"],
                "threshold_profile": row["threshold_profile"],
                "replay_agreement": replay_agreement,
                "control_agreement": control_agreement,
                "leakage_violation": leakage_violation,
                "expected": expected,
                "observed": {
                    "representability": observed_rep,
                    "noncollapse": observed_non,
                },
                "pass_a_sha256": a["canonical_output_sha256"],
                "pass_b_sha256": b["canonical_output_sha256"],
                "missing_required_fields": missing_fields,
                "preservation_status": "PRESERVED_COUNTEREXAMPLE_OR_LIMITATION",
            })

    return completed, falsifications

COMPLETED_ROWS, FALSIFICATIONS = evaluate_results(HELD_OUT_INPUT_ROWS, PASS_A, PASS_B)
print("Completed rows:", len(COMPLETED_ROWS))
print("Falsification flags:", len(FALSIFICATIONS))


## 7. Summary and bounded interpretation

In [ ]:

from collections import Counter

rep_counts = Counter(r["representability_classification"] for r in COMPLETED_ROWS)
non_counts = Counter(r["noncollapse_classification"] for r in COMPLETED_ROWS)
replay_mismatches = [r for r in COMPLETED_ROWS if not r["replay_agreement"]]
control_mismatches = [r for r in COMPLETED_ROWS if not r["control_agreement"]]
leakage_violations = [r for r in COMPLETED_ROWS if r["context_history_leakage_violation"]]

all_pass = not (replay_mismatches or control_mismatches or leakage_violations)
if all_pass:
    bounded_conclusion = (
        "Within the declared finite held-out design, the frozen candidate predicates "
        "replayed exactly and matched every declared control. This supports only bounded "
        "replay consistency for these rows."
    )
else:
    bounded_conclusion = (
        "The declared finite held-out design produced preserved mismatch, control, or "
        "leakage evidence. The affected rows are limitations/counterexamples and block "
        "a bounded consistency support statement."
    )

SUMMARY = {
    "spec_id": SPEC_ID,
    "notebook_id": NOTEBOOK_ID,
    "status": "BOUNDED_SUPPORT" if all_pass else "LIMITATION_OR_NEGATIVE_RESULT",
    "claim_ceiling": EXPERIMENT_SPEC["interpretation_protocol"]["claim_ceiling"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "declared_design": {
        "contexts": list(VALID_CONTEXTS),
        "cases": list(VALID_CASES),
        "witness_modes": list(VALID_WITNESS_MODES),
        "seeds": list(VALID_SEEDS),
        "threshold_profiles": list(VALID_THRESHOLD_PROFILES),
        "repetitions": 2,
    },
    "row_count": len(COMPLETED_ROWS),
    "replay_agreement_count": sum(r["replay_agreement"] for r in COMPLETED_ROWS),
    "replay_mismatch_count": len(replay_mismatches),
    "control_agreement_count": sum(r["control_agreement"] for r in COMPLETED_ROWS),
    "control_mismatch_count": len(control_mismatches),
    "leakage_violation_count": len(leakage_violations),
    "falsification_flag_count": len(FALSIFICATIONS),
    "representability_counts": dict(sorted(rep_counts.items())),
    "noncollapse_counts": dict(sorted(non_counts.items())),
    "bounded_conclusion": bounded_conclusion,
    "allowed_interpretations": EXPERIMENT_SPEC["interpretation_protocol"]["allowed_interpretations"],
    "blocked_interpretations": EXPERIMENT_SPEC["interpretation_protocol"]["blocked_interpretations"],
    "spec_hash_check": SPEC_HASH_CHECK,
}

FALSIFICATION_REPORT = {
    "spec_id": SPEC_ID,
    "status": "NO_FLAGGED_ROWS" if not FALSIFICATIONS else "PRESERVED_FLAGS_PRESENT",
    "claim_ceiling": EXPERIMENT_SPEC["interpretation_protocol"]["claim_ceiling"],
    "flag_count": len(FALSIFICATIONS),
    "flags": FALSIFICATIONS,
    "negative_evidence_policy": "No flagged row may be discarded or reclassified post hoc.",
}

print(json.dumps(SUMMARY, indent=2))


## 8. Write required artifacts and provenance manifest

In [ ]:

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")


def write_jsonl(path: Path, rows: Iterable[Mapping[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False))
            f.write("\n")

held_out_path = RESULT_DIR / "held_out_rows.jsonl"
summary_path = RESULT_DIR / "summary.json"
falsification_path = RESULT_DIR / "falsification_report.json"
manifest_path = RESULT_DIR / "manifest.json"
spec_copy_path = RESULT_DIR / "experiment_spec.embedded.json"

write_jsonl(held_out_path, COMPLETED_ROWS)
write_json(summary_path, SUMMARY)
write_json(falsification_path, FALSIFICATION_REPORT)
write_json(spec_copy_path, EXPERIMENT_SPEC)

artifact_paths = [held_out_path, summary_path, falsification_path, spec_copy_path]
artifact_hashes = {
    str(p.relative_to(ROOT)): sha256_bytes(p.read_bytes()) for p in artifact_paths
}

MANIFEST = {
    "spec_id": SPEC_ID,
    "notebook_id": NOTEBOOK_ID,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "predicate_version": FROZEN_PREDICATE_VERSION,
    "predicate_sha256": FROZEN_PREDICATE_SHA256,
    "blind_expected_control_sha256": BLIND_EXPECTED_SHA256,
    "embedded_spec_full_sha256": SPEC_EMBEDDED_SHA256,
    "declared_spec_content_sha256": EXPERIMENT_SPEC["immutability"]["content_sha256"],
    "spec_hash_check": SPEC_HASH_CHECK,
    "row_count": len(COMPLETED_ROWS),
    "repetitions": 2,
    "artifact_hashes": artifact_hashes,
    "provenance_source_paths": EXPERIMENT_SPEC["provenance"]["source_paths"],
    "interpretation_claim_ceiling": EXPERIMENT_SPEC["interpretation_protocol"]["claim_ceiling"],
    "manifest_hash_policy": "SHA-256 over exact written artifact bytes; manifest excludes its own hash.",
}
write_json(manifest_path, MANIFEST)

print("Written artifacts:")
for p in [held_out_path, summary_path, falsification_path, manifest_path, spec_copy_path]:
    print(" -", p.resolve(), p.stat().st_size, "bytes")


## 9. Final integrity gates

In [ ]:

required_paths = [
    RESULT_DIR / "held_out_rows.jsonl",
    RESULT_DIR / "summary.json",
    RESULT_DIR / "falsification_report.json",
    RESULT_DIR / "manifest.json",
]
assert all(p.exists() and p.stat().st_size > 0 for p in required_paths)
assert len(COMPLETED_ROWS) == EXPECTED_ROW_COUNT
assert all(r["replay_agreement"] for r in COMPLETED_ROWS), "Replay mismatch preserved in outputs."

# Control mismatches are not erased. This assertion makes a normal clean run
# visibly fail after artifacts have already been written if a mismatch exists.
assert not [r for r in COMPLETED_ROWS if not r["control_agreement"]], (
    "One or more expected-control mismatches were preserved in falsification_report.json"
)
assert not [r for r in COMPLETED_ROWS if r["context_history_leakage_violation"]], (
    "One or more field-binding/leakage violations were preserved in falsification_report.json"
)

print("PASS: all declared combinations completed, replayed exactly, matched controls, and wrote required artifacts.")
print("Claim remains bounded by:", EXPERIMENT_SPEC["interpretation_protocol"]["claim_ceiling"])



## Interpretation reminder

A clean run supports only this statement:

> Within the declared finite held-out contexts and controlled source-relation cases, the frozen candidate predicates produced deterministic replay classifications matching the predeclared control matrix.

It does not establish universal D/E semantics, universal source-relation preservation, injectivity, reversibility, theorem promotion, or external physical validity.
